# 03 — Learn from the past without learning from the future

**Plain-language question:** Which rows may teach and choose the model, and
which rows must remain untouched?

**Why this matters:** using future information can make an offline model look
excellent even though that information will not exist when predictions are
needed.

**Estimated time:** 50–60 minutes.
**Prerequisite:** lessons 00–02; you understand prediction time, labels, data
quality checks, and time cohorts.


## Preflight

Run the environment check, then load development data only.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import (
    add_intentional_leakage,
    load_split,
)
from aai_local_classification.modeling import feature_frame
from aai_local_classification.workflow import ensure_prepared

manifest = ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
print("✓ Loaded train and validation; test labels remain unopened")


### What you should see

A success message naming only train and validation. The manifest can describe
the test file without loading its rows.

### Words introduced

| Word | Plain meaning | Single job in this course |
|---|---|---|
| training split | Rows used to learn model parameters | fit preprocessing/model |
| validation split | Later rows used to choose among fixed options | model + threshold choice |
| frozen test split | Still-later rows reserved as a final exam | one release decision |


## See the time boundaries before writing model code

```text
Jan 2023 ───────── Jun 2024 | Jul ─ Sep 2024 | Oct ─ Dec 2024
          TRAIN             |   VALIDATION   |   FROZEN TEST
      learn parameters      | choose once    | judge fixed choice
```

**Before you run this:** predict whether any split's dates should overlap.


In [ ]:
split_calendar = pd.DataFrame(
    [
        {
            "split": item.split.value,
            "rows": item.row_count,
            "start": item.start_date,
            "end": item.end_date,
            "label_rate_visible": item.positive_rate is not None,
        }
        for item in manifest.artifacts
    ]
)
split_calendar


### How to interpret the output

Training has 2,160 rows, validation 360, and test 360. Dates are ordered and do
not overlap. The final row says the test label rate is not visible. Think of the
test split as a sealed final exam, not extra practice questions.


In [ ]:
ordered_boundaries = pd.Series(
    {
        "train_ends_before_validation": (
            train.snapshot_date.max() < validation.snapshot_date.min()
        ),
        "validation_ends_before_test": (
            validation.snapshot_date.max() < pd.Timestamp(settings.data.test_start)
        ),
    },
    name="passed",
)
ordered_boundaries.to_frame()


### What you should see

Both checks are `True`. A random split is not automatically safer: the split
should imitate how the model will encounter future data. This generator has one
snapshot per synthetic account; a real dataset with repeated accounts would
also need an entity-aware design.


## Make the model inputs and labels explicit

Uppercase `X` conventionally means the feature table; lowercase `y` means the
target series. They are ordinary pandas objects.

**Before you run this:** there are 9 declared model features. Predict the two
shapes.


In [ ]:
X_train = feature_frame(train, settings)
y_train = train[settings.data.target_column]
X_validation = feature_frame(validation, settings)
y_validation = validation[settings.data.target_column]

pd.Series(
    {
        "X_train": X_train.shape,
        "y_train": y_train.shape,
        "X_validation": X_validation.shape,
        "y_validation": y_validation.shape,
    }
).to_frame("shape")


### What you should see

`X_train` is `(2160, 9)` while `y_train` is `(2160,)`; validation has 360 rows
in both objects. `account_id`, `snapshot_date`, and `churned_30d` are absent
from `X` by contract.


## Leakage: when the answer sneaks into the inputs

### Words introduced

| Word | Plain meaning | Example |
|---|---|---|
| target leakage | An input directly reveals the later answer | cancellation reason |
| temporal leakage | Training uses information from after prediction time | future support tickets |
| preprocessing leakage | A cleanup step learns from validation/test | scaling on all rows |

The next function creates a teaching-only column. It does not alter persisted
data.


In [ ]:
leaked_train = add_intentional_leakage(train)
pd.crosstab(
    leaked_train.cancellation_reason,
    leaked_train.churned_30d,
    rownames=["post-outcome value"],
    colnames=["true label"],
)


### How to interpret the output

`account_closed` occurs only when the label is `1`; `not_applicable` occurs only
when it is `0`. A model could appear perfect by reading the answer after it
happened. At real prediction time the column would not exist, so the model would
fail operationally.


## Measure the leak instead of trusting the story

A crosstab raises suspicion; a fit turns it into a number. The next cell trains
the same preprocessing + logistic-regression pipeline twice on the same
training rows: once on the nine honest features, and once with
`cancellation_reason` smuggled in as a tenth categorical feature. Building the
leaky twin requires hand-crafting a settings object that skips the feature
contract — which is exactly how leaks happen in real projects: a private code
path bypasses the guard.

### Words introduced

| Word | Plain meaning | Single job in this course |
|---|---|---|
| average precision (AP) | Ranking-quality score; no-skill sits near the churn rate, perfect is 1.0 | compare models on one number |

Lesson 04 builds classification metrics carefully. Here you only need: higher
is better, and a perfect `1.00` should make you suspicious, not proud.

**Before you run this:** predict both validation numbers. Honest churn
prediction is genuinely hard, so expect a middling honest score. Where does
the leaky twin land?


In [ ]:
from aai_local_classification.contracts import FeatureSettings
from aai_local_classification.modeling import build_candidate, candidate_specs

leaky_features = FeatureSettings(
    numeric=settings.features.numeric,
    categorical=settings.features.categorical + ("cancellation_reason",),
    forbidden=settings.features.forbidden,
)
leaky_settings = settings.model_copy(update={"features": leaky_features})
logistic_spec = candidate_specs()[0]
honest_model = build_candidate(logistic_spec, settings).fit(X_train, y_train)
leaky_model = build_candidate(logistic_spec, leaky_settings).fit(
    feature_frame(leaked_train, leaky_settings), y_train
)
print("✓ Fitted an honest pipeline and a leaky twin on the same training rows")


In [ ]:
from sklearn.metrics import average_precision_score

leaked_validation = add_intentional_leakage(validation)


def validation_average_precision(model, features):
    positive = list(model.classes_).index(1)
    scores = model.predict_proba(features)[:, positive]
    return average_precision_score(y_validation, scores)


honest_ap = validation_average_precision(honest_model, X_validation)
leaky_ap = validation_average_precision(
    leaky_model, feature_frame(leaked_validation, leaky_settings)
)
print(f"Honest features:          validation AP = {honest_ap:.2f}")
print(f"With cancellation_reason: validation AP = {leaky_ap:.2f}")


### What you should see

Close to `0.47` for the honest features and a perfect `1.00` for the leaky
twin. The cells print two decimals because trailing digits can differ across
platforms; the gap between the two numbers is the point.

### How to interpret the output

One glance at a leaderboard would crown the leaky model. Nothing in the number
itself warns you: the inflation comes from `cancellation_reason` restating the
label, so the model earns a perfect ranking by reading the answer sheet.
Offline evaluation cannot detect this by itself — you must know each feature's
timing.


## The perfect model collapses at prediction time

At serving time the business asks: which current subscribers look likely to
churn *next month*? Nobody has a cancellation reason yet, so the column cannot
exist in a future scoring frame.

**Before you run this:** predict what happens when the leaky model scores the
honest validation features, which lack the leaked column.


In [ ]:
future_frame = X_validation  # at scoring time, cancellation_reason cannot exist
try:
    leaky_model.predict_proba(future_frame)
    serving_result = "Scored without error — the leak would go unnoticed"
except ValueError as error:
    serving_result = f"ValueError: {error}"
print("Serving the leaky model on a frame without the leaked column:")
print(serving_result)


### What you should see

A printed `ValueError` naming the missing `cancellation_reason` column. The
"perfect" model cannot score a single future row. This failure mode is the
*lucky* one: a loud crash on the first real scoring run. The unlucky version is
a leaked column that still exists at serving time with a different meaning —
silently wrong scores instead of an error.


The configuration explicitly lists forbidden fields. Packaged training runs
this readable rule before any expensive fit — the leaky twin above could only
exist because we deliberately bypassed that path.


In [ ]:
forbidden_attempt = "cancellation_reason"
contract_result = pd.Series(
    {
        "attempted_feature": forbidden_attempt,
        "listed_as_forbidden": forbidden_attempt in settings.features.forbidden,
        "allowed_to_train": forbidden_attempt not in settings.features.forbidden,
    }
)
contract_result.to_frame("value")


### What you should see

The attempted field is listed as forbidden and `allowed_to_train` is `False`.
Packaged training also calls `validate_feature_contract`, so a source change
cannot silently bypass this display.

### Misconception check

“The column improves validation” is not enough. First ask whether its value
exists, with the same meaning, at the exact prediction time.


## Preprocessing leakage: the quiet third kind

Target leakage shouts once you measure it. Preprocessing leakage whispers: no
column is wrong, yet a cleanup step fitted on all rows has absorbed information
from the rows that are supposed to judge the model. Compare a `StandardScaler`
fitted only on training rows with one fitted on training plus validation rows.

**Before you run this:** monthly fees drift upward over time, and validation
rows are later than training rows. Predict which fitted mean is larger.


In [ ]:
from sklearn.preprocessing import StandardScaler

train_only_scaler = StandardScaler().fit(X_train[["monthly_fee"]])
peeking_scaler = StandardScaler().fit(
    pd.concat([X_train, X_validation], ignore_index=True)[["monthly_fee"]]
)
scaler_comparison = pd.DataFrame(
    {
        "fitted_mean": [train_only_scaler.mean_[0], peeking_scaler.mean_[0]],
        "fitted_scale": [train_only_scaler.scale_[0], peeking_scaler.scale_[0]],
    },
    index=["train only", "train + validation"],
)
scaler_comparison.round(2)


### What you should see

Two different scalers: fitted means near `64.23` and `64.85`, with a smaller
difference in scale. The peeking scaler drifted toward validation's later,
higher fees.

### How to interpret the output

The shift looks small, but it means every “standardized” training value was
computed using information from the evaluation period. This is why the course
keeps the scaler inside one sklearn `Pipeline`: calling `fit` on training rows
fits the scaler and the model on exactly the same rows, and validation rows
only ever pass through `transform`.


### Guided exercise

Classify “support tickets opened during the 30 days after the snapshot.” Is it
available or forbidden for this prediction? Change the starter answer if needed.


In [ ]:
exercise_feature_timing = "forbidden"
exercise_explanation = "Those tickets happen after prediction time."
pd.Series(
    {
        "classification": exercise_feature_timing,
        "explanation": exercise_explanation,
    }
)


**Self-check:** only tickets observed before the monthly snapshot could be
eligible. A similarly named 90-day history feature is allowed because its
window ends at the snapshot.

<details><summary>Solution explanation</summary>

The future 30-day ticket count is forbidden temporal leakage. The existing
`support_tickets_90d` feature is a backward-looking value available now.
</details>


In [ ]:
# Reference solution — run after your attempt
assert exercise_feature_timing == "forbidden"
assert "after" in exercise_explanation.lower()
print("✓ Future-window support data is excluded")


## MLOps bridge

The exact date predicates and dataset fingerprints become run evidence. On
Databricks, a reviewed job would read versioned Delta data using the same time
boundaries; changing model or policy after a test result requires a new frozen
test version.

## Recap

- Train learns parameters, validation chooses, and frozen test judges the fixed
  choice once.
- A production-like time split is more meaningful than a convenient random
  split for this scenario.
- Feature availability is evaluated at prediction time; future facts are
  leakage even when present in historical storage.
- The leak was measured, not narrated: the leaky twin scored a perfect
  validation average precision, then could not score a future frame at all.
- A scaler fitted on training plus validation rows quietly absorbs future
  information; fitting preprocessing inside one pipeline on train only
  prevents it.

**Evidence created:** explicit `X_train`, `y_train`, `X_validation`, and
`y_validation`, plus two throwaway teaching models, all in memory only. No
test rows were loaded and nothing was persisted.

**Ready for 04?** You can give one sentence for each split's job and identify
the three leakage types above.
